# 서울 공공자전거 따릉이 수요 예측
### 시계열 모델 기반 정류소 재배치 전략

**Group 2** | 2026 Spring Time Series Analysis

---
> **[발표 순서]**  
> 1. 서론 — 문제 정의, 평가 지표  
> 2. 데이터 — 출처, 결측치, 2023+ 선택  
> 3. EDA — S=24 / 평일·주말 분리 / 외생변수 근거  
> 4. 모델링 — SARIMA, ARIMAX, RF, XGBoost  
> 5. 결과 비교  
> 6. Limitation & 결론

In [ ]:
# ── 공통 세팅 (발표 전 미리 실행) ──────────────────────────────
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.figsize'] = (12, 4)

from src.config import DATA_DIR, TARGET_RENT_IDS, TARGET_RENT_ID
from src.utils import load_filtered_csvs, make_series, load_weather
try:
    from src.utils import load_weather_full
    weather_df = load_weather_full(DATA_DIR)
    HAS_RAIN = True
except Exception:
    weather_df = load_weather(DATA_DIR)
    HAS_RAIN = False

df_raw  = load_filtered_csvs(DATA_DIR)   # 전체 기간
df_2023 = df_raw[df_raw['datetime'].dt.year >= 2023]
series  = make_series(df_2023, TARGET_RENT_ID)   # 대표 정류소 02128

print('데이터 로드 완료')

---
## 1. 서론
> **[발표]** 따릉이 재배치 문제를 소개하고, 왜 예측이 필요한지 설명한다.  
> 핵심: *자전거가 없으면 아예 못 빌린다 → 과소예측이 더 큰 손해*

### 문제 정의

- **목표**: 23개 정류소 시간별 대여 수요 예측 → 재배치 전략 수립
- **핵심 지표**: Asymmetric RMSE (α=2) — 과소예측(자전거 고갈)에 **2배 패널티**

$$L(\hat{y}, y) = \begin{cases} \alpha \cdot (\hat{y} - y)^2 & \hat{y} < y \ (\text{과소예측}) \\ (\hat{y} - y)^2 & \hat{y} \geq y \ (\text{과대예측}) \end{cases}$$

> **왜 α=2?** 자전거가 없으면 수요 자체가 censored — 실제 수요보다 항상 낮게 관측됨

---
## 2. 데이터
> **[발표]** 데이터 출처 → 결측치 처리 → 왜 2023+만 쓰는지 순서로 설명

### 데이터 출처

| 데이터 | 출처 | 내용 |
|--------|------|------|
| 대여이력 | 서울 열린데이터광장 OA-15182 | 시간별 정류소 대여 건수 |
| 기상 | 기상청 서울 관측소 (108번) | 시간별 기온, 강수량 |

- 분석 대상: 서울대 인근 **23개 정류소**, 2023.01 ~ 2024.12
- 집계 단위: **1시간**

In [ ]:
# ── 결측치 현황 ────────────────────────────────────────────────────
# [발표] "결측률 2.1% — 하루 전체가 비어있을 때만 결측으로 판정"
#        "심야 0건은 정상 데이터, 결측 아님"

df_raw['date'] = df_raw['datetime'].dt.date
all_dates = pd.date_range(df_raw['datetime'].min().date(),
                           df_raw['datetime'].max().date(), freq='D')
total_days = len(all_dates)

miss_data = []
for rid in TARGET_RENT_IDS:
    obs = df_raw[df_raw['RENT_ID'] == rid]['date'].nunique()
    miss_data.append({'station': rid, 'pct': (total_days - obs) / total_days * 100})
miss_df = pd.DataFrame(miss_data).sort_values('pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(miss_df['station'], miss_df['pct'],
       color=['tomato' if p > 5 else 'steelblue' for p in miss_df['pct']])
ax.axhline(2.1, color='gray', linestyle='--', linewidth=1, label='평균 2.1%')
ax.set_ylabel('결측률 (%)')
ax.set_title('정류소별 결측률 (하루 전체 결측 기준) — 전체 평균 2.1%')
ax.legend()
plt.xticks(rotation=45, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2023+ 선택 근거 — COVID 기간 수요 이상 ────────────────────────
# [발표] "2021~2022는 COVID로 수요가 비정상적으로 낮음"
#        "코로나 이후 패턴을 학습해야 2025년 예측에 유효"

monthly = (df_raw.assign(ym=df_raw['datetime'].dt.to_period('M'))
           .groupby('ym')['CNT'].sum().reset_index())
monthly['dt'] = monthly['ym'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['dt'], monthly['CNT'], color='steelblue', linewidth=1.5)
ax.fill_between(monthly['dt'], monthly['CNT'], alpha=0.15, color='steelblue')
ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2022-12-31'),
           alpha=0.15, color='red', label='COVID 기간 (제외)')
ax.axvline(pd.Timestamp('2023-01-01'), color='red', linestyle='--',
           linewidth=2, label='학습 시작점')
ax.set_title('월별 총 대여량 — 2023년부터 정상 수요 회복')
ax.set_ylabel('월별 대여량 (23개 정류소 합계)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 3. EDA — 모델링 결정 근거
> **[발표]** 그래프 보여주면서 "이래서 S=24 고정", "이래서 평일/주말 분리" 순서로 설명

In [ ]:
# ── 평일 vs 주말 시간대 패턴 → 분리 근거 ─────────────────────────
# [발표] "평일은 출퇴근 더블 피크, 주말은 오후 단일 피크 — 하나의 모델로 못 잡음"

df_pat = pd.DataFrame({'CNT': series})
df_pat['hour'] = df_pat.index.hour
df_pat['is_weekend'] = df_pat.index.dayofweek >= 5
df_pat['dayofweek'] = df_pat.index.dayofweek
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for is_wknd, label, color in [(False,'Weekday','steelblue'),(True,'Weekend','tomato')]:
    h = df_pat[df_pat['is_weekend']==is_wknd].groupby('hour')['CNT'].mean()
    axes[0].plot(h.index, h.values, label=label, color=color, linewidth=2.5, marker='o', markersize=3)
axes[0].set_title('시간대별 평균 수요 — 평일 vs 주말\n→ 구조적으로 다른 패턴')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Average CNT')
axes[0].set_xticks(range(0,24,2))
axes[0].legend()

daily = df_pat.groupby('dayofweek')['CNT'].mean()
colors = ['steelblue']*5 + ['tomato']*2
axes[1].bar([dow_names[i] for i in daily.index], daily.values, color=colors)
axes[1].set_title('요일별 평균 수요\n→ 평일/주말 분리 정당화')
axes[1].set_ylabel('Average hourly CNT')

plt.tight_layout()
plt.show()

In [ ]:
# ── ACF → S=24 정당화 ──────────────────────────────────────────────
# [발표] "lag 24, 48, 72에서 반복되는 스파이크 → 24시간 주기 계절성 명확"

s_wd = series[series.index.dayofweek < 5]

fig, ax = plt.subplots(figsize=(13, 3.5))
plot_acf(s_wd.dropna(), lags=72, ax=ax,
         title='ACF — 평일 시계열 | lag 24, 48, 72에서 유의한 피크 → S=24')
for lag in [24, 48, 72]:
    ax.axvline(lag, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.text(24, ax.get_ylim()[1]*0.85, 'lag 24', color='red', fontsize=9)
ax.text(48, ax.get_ylim()[1]*0.85, 'lag 48', color='red', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 기상 외생변수 근거 ─────────────────────────────────────────────
# [발표] "기온-수요 2차 관계: 너무 추워도 안 타고, 너무 더워도 안 탐"
#        "비 오는 날은 수요 명확히 감소"

df_w = pd.DataFrame({'CNT': series})
df_w['temp'] = weather_df['temp'].reindex(df_w.index).ffill()
if HAS_RAIN:
    rc = 'rainfall' if 'rainfall' in weather_df.columns else 'rain'
    df_w['rain'] = weather_df[rc].reindex(df_w.index).fillna(0)
    df_w['is_rain'] = df_w['rain'] > 1.0
df_w['month'] = df_w.index.month
df_w = df_w.dropna()

ncols = 3 if HAS_RAIN else 2
fig, axes = plt.subplots(1, ncols, figsize=(13, 4))

samp = df_w.sample(min(3000, len(df_w)), random_state=42)
axes[0].scatter(samp['temp'], samp['CNT'], alpha=0.08, s=5, color='steelblue')
z = np.polyfit(samp['temp'], samp['CNT'], 2)
tr = np.linspace(samp['temp'].min(), samp['temp'].max(), 100)
axes[0].plot(tr, np.poly1d(z)(tr), color='red', linewidth=2)
axes[0].set_xlabel('기온 (°C)')
axes[0].set_ylabel('대여량')
axes[0].set_title('기온 vs 대여량 (2차 관계)')

m_avg = df_w.groupby('month')[['CNT','temp']].mean()
ax2b = axes[1].twinx()
axes[1].bar(m_avg.index, m_avg['CNT'], color='steelblue', alpha=0.6, label='수요')
ax2b.plot(m_avg.index, m_avg['temp'], color='red', marker='o', linewidth=2, label='기온')
axes[1].set_xticks(range(1,13))
axes[1].set_xlabel('월')
axes[1].set_ylabel('평균 대여량', color='steelblue')
ax2b.set_ylabel('평균 기온', color='red')
axes[1].set_title('월별 수요 & 기온')

if HAS_RAIN:
    ra = df_w.groupby('is_rain')['CNT'].mean()
    bars = axes[2].bar(['비 없음','비 있음\n(>1mm)'], ra.values, color=['steelblue','gray'])
    for b, v in zip(bars, ra.values):
        axes[2].text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}',
                     ha='center', va='bottom', fontsize=11, fontweight='bold')
    axes[2].set_ylabel('평균 대여량')
    axes[2].set_title('강수 여부별 수요')

plt.tight_layout()
plt.show()

---
## 4. 모델링
> **[발표]** 각 모델을 한 슬라이드씩. 수식보다 직관적 설명 위주.  
> SARIMA → ARIMAX (외생변수 추가) → RF → XGBoost (비대칭 손실) 흐름으로

### 모델 구성

| 모델 | 특징 | 외생변수 |
|------|------|----------|
| **SARIMA** `(2,0,1)×(1,1,0,24)` | 통계 시계열, S=24 계절성 | ✗ |
| **ARIMAX** `(2,0,1)×(1,1,0,24)` | SARIMA + 기온·강수 | ✓ |
| **Random Forest** | 트리 앙상블, lag/시간 피처 | ✓ |
| **XGBoost** | Gradient Boosting | ✓ |
| **XGBoost (α=2)** | 비대칭 손실 — 과소예측 2× 패널티 | ✓ |

**ML 공통 피처**: `hour`, `dayofweek`, `month`, `lag_1`, `lag_24`, `lag_168`, `rolling_mean_24/168`, `temp`, `rain`

```python
# XGBoost 비대칭 손실 핵심 코드
def asymmetric_obj(y_pred, dtrain, alpha=2.0):
    errors = y_pred - dtrain.get_label()
    grad = np.where(errors < 0, alpha * 2 * errors, 2 * errors)  # 과소예측에 α배
    hess = np.where(errors < 0, alpha * 2, 2)
    return grad, hess
```

---
## 5. 결과 비교
> **[발표]** 표 먼저 보여주고 → 그래프로 시각화  
> "RMSE만 보면 ARIMAX/XGBoost가 좋지만, 과소예측률 보면 ARIMAX가 46%로 최악"  
> "Asym.RMSE + 과소예측률 동시에 낮은 게 XGBoost(α=2)"

In [ ]:
# ── 모델 성능 비교 (station 02128, 평일) ──────────────────────────
# [발표] 표를 먼저 읽어주고, 아래 그래프로 시각적으로 설명

results = pd.DataFrame([
    {'model': 'SARIMA',           'RMSE': 0.640, 'Asym_RMSE': 0.750, 'Under%': 26.2},
    {'model': 'ARIMAX',           'RMSE': 0.599, 'Asym_RMSE': 0.790, 'Under%': 46.2},
    {'model': 'Random Forest',    'RMSE': 0.601, 'Asym_RMSE': 0.762, 'Under%': 27.9},
    {'model': 'XGBoost',          'RMSE': 0.599, 'Asym_RMSE': 0.763, 'Under%': 27.9},
    {'model': 'XGBoost (α=2)',    'RMSE': 0.650, 'Asym_RMSE': 0.761, 'Under%': 26.4},
]).set_index('model')

print('=== Station 02128 평일 성능 비교 ===')
print(results.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue','steelblue','steelblue','steelblue','tomato']

results['RMSE'].plot(kind='bar', ax=axes[0], color=colors, rot=20)
axes[0].set_title('RMSE\n(낮을수록 좋음)')
axes[0].set_ylim(0.55, 0.70)

results['Asym_RMSE'].plot(kind='bar', ax=axes[1], color=colors, rot=20)
axes[1].set_title('Asymmetric RMSE α=2\n★ 주요 지표')
axes[1].set_ylim(0.70, 0.82)

results['Under%'].plot(kind='bar', ax=axes[2], color=colors, rot=20)
axes[2].set_title('과소예측률 (%)\n(낮을수록 좋음)')
axes[2].set_ylim(0, 55)
axes[2].axhline(30, color='gray', linestyle='--', linewidth=0.8)

for ax in axes:
    ax.set_xlabel('')

plt.suptitle('Station 02128 평일 모델 성능 비교', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Limitation & 결론
> **[발표]** 한계점 솔직하게 인정하고, 향후 방향 제시로 마무리

### Limitation

1. **Demand Censoring**  
   자전거 재고가 없으면 관측 수요 < 실제 수요. 비대칭 손실로 부분 완화했으나 근본 해결 아님.

2. **공통 차수 적용**  
   auto_arima를 대표 정류소에서 찾아 23개에 동일 적용. 정류소별 최적 차수는 다를 수 있음.

3. **잔차 ACF 미해소**  
   lag 24, 48에 자기상관 잔존 — 주간 계절성(S=168)이 완전히 반영되지 않음.

---

### 결론

| | |
|--|--|
| **최선 모델** | XGBoost (비대칭 손실 α=2) |
| **이유** | Asym.RMSE 최저 + 과소예측률 균형 |
| **향후** | 23개 정류소 예측 → 재배치 권고 전략 수립 |